<img src="logo.png" alt="Vegeta" width="240">

# Air propellers — tractor and pusher, two and three blades: efficiency, CFD, flow movies, blade FEA, noise

One 10 × 6 inch propeller on the motor pod of a small fixed-wing drone, the pod hanging from a pylon (a wing root or
a strut). Four configurations, the **tractor first** (propeller ahead of the pod: it pulls, and its blades meet
almost undisturbed air), then the **pusher** (propeller behind the pod: it pushes, and every blade cuts through the
pylon's wake once per revolution), each with **two and three blades** of the same blade:

```
1  the design and the flight points (edit here)
2  aerodynamic efficiency (Boreas BEMT): η, C_T, C_P against J; efficiency across the flight envelope; loading along the blade
3  the air the blades meet: the pylon's wake (pusher) and its blockage (tractor), its harmonics
4  the load on a blade over one revolution — what the wake does to it
5  CFD (OpenFOAM): the isolated propellers (rotor_mrf) and the installed ones (rotor_mrf_installed: rotating blades,
   standing pod and pylon, forces apart), each over an rpm sweep at cruise speed; the pod alone; the flow in sections
6  efficiency against rpm — six propellers: alone, tractor, pusher × two and three blades (model at five airspeeds,
   where the aircraft flies, the CFD sweep)
7  the flow in 3D with particles
8  movies: tracer balls through the four configurations (tractor 2 → tractor 3 → pusher 2 → pusher 3), from the CFD
9  the blade in FEA (Talos): spinning (centrifugal) + aerodynamic load, weak points, modes, Campbell diagram, stress movie
10 vibration and fatigue: the pusher's once-per-revolution load pulse against the blade's strength
11 how loud: tones (steady + unsteady blade loading, Lowson's rotating dipoles), broadband, dB and dB(A), directivity,
   and the sound itself — four WAV files at one common scale
12 side by side, and export
13 export — the results the ducted-fan (25b) and exotic-propeller (25c) notebooks compare against, as JSON
14 the propeller maps for the flight models (MERLIN, notebooks 26b and 27)
```

**CFD presets.** The OpenFOAM cells run a **smoke test** by default (`AIR_PROP_CFD=smoke`: coarse meshes, 60
iterations, about 20 s per case on 4 cores) — it proves the cases mesh, run and post-process, and its numbers are
**not results**. For the real run set `AIR_PROP_CFD=full` (finer meshes, 600 iterations; several hours on 4 cores, not
timed here; `AIR_PROP_PROCS` processes), which also uses the pod-alone wake for the noise. Both presets run every propeller
at three rpm — **low, medium, high** (`RPM_POINTS`: 0.85, 1.0 and 1.25 × its cruise rpm) — at the cruise speed: 18
propeller cases, 2 pod-alone cases and 3 ducted-fan cases (section 13). `VEGETA_SKIP_OPENFOAM=1` skips the CFD (the model-based sections still run).

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, shutil, sys, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv
from tqdm.auto import tqdm
from IPython.display import Video, Audio, display
from vegeta import dedalus, talos, aeromant, boreas
from vegeta.boreas import wake
from vegeta.aeromant import movie
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.dedalus.examples import Propeller as PropellerCAD
sys.path.insert(0, "designs")
import air_propeller as ap

RUNS = Path("_runs/air_propeller"); shutil.rmtree(RUNS, ignore_errors=True); (RUNS / "cad").mkdir(parents=True)
OUT = Path("output/25_air_propeller"); OUT.mkdir(parents=True, exist_ok=True)       # every movie and sound of this notebook
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
CFD_PRESET = os.environ.get("AIR_PROP_CFD", "smoke")       # "smoke": seconds per case, proves the pipeline; "full": the real run
PROCS = int(os.environ.get("AIR_PROP_PROCS", "4"))
assert CFD_PRESET in ("smoke", "full"), CFD_PRESET
RHO, NU, G = 1.225, 1.5e-5, 9.81                          # sea-level air, gravity
AIR = boreas.Medium("air", RHO, 340.0)
print(f"CFD: {'on' if RUN_CFD else 'skipped'}, preset {CFD_PRESET!r}, {PROCS} processes")

## 1. The design and the flight points — edit here

`BLADE` is the blade (the same for two and three blades: a 10 × 6 inch electric-drone propeller), `SECTION` its
aerofoil (explicit polar inputs), `HUB` the hub. `POD` is the motor pod and its pylon (`designs/air_propeller.py`);
the pylon's nearer edge is `pylon_gap` from the propeller plane in both layouts, so only the side changes.
`AIRCRAFT` is the drone the propeller flies (a 3 kg fixed wing: its drag polar sets the thrust needed at every speed).
`MATERIAL` is a moulded glass-filled nylon blade (assumed values; replace with the supplier's data).

In [ ]:
BLADE = dict(diameter_mm=254.0, pitch_mm=152.4, chord_root_mm=18.0, chord_max_mm=26.0, chord_tip_mm=10.0, thickness=0.10, camber=0.04)
HUB = dict(hub_diameter=20.0, hub_height=10.0, bore=5.0)
SECTION = boreas.Airfoil(name="thin cambered section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-3.0, cl_max=1.1, cd0=0.018, k=0.04,
                         source="assumed for a 10-inch blade at Re ~ 1e5 (fit a measured polar here)")
POD = dict(pod_diameter=46.0, pod_length=240.0, pylon_chord=90.0, pylon_thickness=0.12, pylon_height=230.0, pylon_gap=64.0)
PYLON_CD = 0.012                          # the pylon section's drag coefficient (sets its wake: an input)
AIRCRAFT = dict(mass_kg=3.0, wing_area_m2=0.50, cd0=0.035, aspect_ratio=8.0, oswald=0.8)
MATERIAL = dict(name="PA6-GF30 (moulded)", E=8000.0, nu=0.35, rho=1.35e-9, yield_=100.0, ultimate=150.0, fatigue_1e7=30.0, damping=0.02)
BLADE_COUNTS = (2, 3)
LAYOUTS = ("tractor", "pusher")
CONFIGS = [(lay, B) for lay in LAYOUTS for B in BLADE_COUNTS]           # tractor first, then pusher
LABEL = {c: f"{c[0]}, {c[1]} blades" for c in CONFIGS}
# one hue per blade count — 2 blue, 3 purple (6 green and 12 orange in notebook 25c; red is the ducted fan's);
# the tractor in the dark shade, the pusher in the light one
COLOR = {("tractor", 2): "#1f4e9c", ("pusher", 2): "#6fa8dc", ("tractor", 3): "#5b2c83", ("pusher", 3): "#b39ddb"}

def required_thrust(V, climb_rate=0.0):
    """Thrust the aircraft needs at airspeed V [m/s] (level flight or a steady climb): drag polar + weight component."""
    a = AIRCRAFT
    W = a["mass_kg"] * G
    if V <= 0:
        return 0.75 * W                                                  # take-off / hand launch: static thrust 75 % of the weight
    q = 0.5 * RHO * V**2
    cl = W / (q * a["wing_area_m2"])
    k = 1 / (math.pi * a["aspect_ratio"] * a["oswald"])
    return q * a["wing_area_m2"] * (a["cd0"] + k * cl**2) + W * climb_rate / V

POINTS = {"cruise": (20.0, 0.0), "climb": (14.0, 3.0), "take-off": (0.0, 0.0)}   # airspeed [m/s], climb rate [m/s]

def prop_of(B):
    return boreas.Propeller.from_pitch(f"10x6, {B} blades", BLADE["diameter_mm"] / 1000, BLADE["pitch_mm"] / 1000, blades=B,
                                       chord_root_m=BLADE["chord_root_mm"] / 1000, chord_max_m=BLADE["chord_max_mm"] / 1000,
                                       chord_tip_m=BLADE["chord_tip_mm"] / 1000, hub_radius_m=HUB["hub_diameter"] / 2000)

def cad_kw(B):
    return dict(diameter=BLADE["diameter_mm"], pitch=BLADE["pitch_mm"], blades=B, chord_root=BLADE["chord_root_mm"],
                chord_max=BLADE["chord_max_mm"], chord_tip=BLADE["chord_tip_mm"], thickness=BLADE["thickness"], camber=BLADE["camber"],
                stations=8, **HUB)

PROPS = {B: prop_of(B) for B in BLADE_COUNTS}
OPS = {B: {k: boreas.rpm_for_thrust(PROPS[B], SECTION, required_thrust(V, cr), V, RHO) for k, (V, cr) in POINTS.items()} for B in BLADE_COUNTS}
pd.DataFrame({(B, k): {"airspeed_m_s": op.airspeed, "thrust_N": op.thrust, "rpm": op.rpm, "shaft_power_W": op.power, "J": op.advance_ratio,
                       "efficiency η": op.efficiency, "figure_of_merit": op.figure_of_merit, "tip_Mach": op.tip_mach}
              for B in BLADE_COUNTS for k, op in OPS[B].items()}).round(3)

The four configurations in CAD: the propeller (Dedalus `Propeller`, the axis turned to +x) and the pod with its pylon
(`PropPod`, standing, the pylon up along +y). The propeller turns right-handed about +x (seen from behind: clockwise
from the pilot's seat of a tractor), the same in all four.

In [ ]:
PROP_STL, PROP_MESH = {}, {}
for B in BLADE_COUNTS:
    cad = PropellerCAD().generate(**cad_kw(B))
    px = dedalus.Geometry.from_cadquery(cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name=f"prop{B}_axis_x")
    PROP_STL[B] = px.export_stl(RUNS / "cad" / f"prop{B}_axis_x.stl", tolerance=0.03)
    PROP_MESH[B] = dviz.to_pyvista(px, 0.05).scale(0.001, inplace=False)
# the pod starts (tractor) or ends (pusher) a clearance from the propeller's real extent: the blade roots overhang the hub
from vegeta.aeromant.stl import read_stl
EXT = [read_stl(PROP_STL[B]).bbox for B in BLADE_COUNTS]                    # mm, axis +x
X_AFT, X_FWD = max(b[1][0] for b in EXT), min(b[0][0] for b in EXT)
CLEARANCE_MM = 2.0
POD_GAP = {"tractor": X_AFT - HUB["hub_height"] / 2 + CLEARANCE_MM, "pusher": -X_FWD - HUB["hub_height"] / 2 + CLEARANCE_MM}
def pod_p(lay):
    return dict(POD, layout=lay, hub_height=HUB["hub_height"], hub_diameter=HUB["hub_diameter"], gap=POD_GAP[lay])

POD_GEOM, POD_STL, POD_MESH = {}, {}, {}
for lay in LAYOUTS:
    POD_GEOM[lay] = ap.PropPod().generate(**pod_p(lay))
    POD_STL[lay] = POD_GEOM[lay].export(RUNS / "cad" / f"pod_{lay}", formats=("stl", "step"), stl_tolerance=0.1).artifacts["stl"]
    POD_MESH[lay] = dviz.to_pyvista(POD_GEOM[lay], 0.2).scale(0.001, inplace=False)
OUTLINE = {lay: ap.outline(pod_p(lay)) for lay in LAYOUTS}
bb = {lay: read_stl(POD_STL[lay]).bbox for lay in LAYOUTS}
assert bb["tractor"][0][0] > X_AFT and bb["pusher"][1][0] < X_FWD, "the propeller would strike the pod"
print(f"propeller spans x = {X_FWD:.2f} .. {X_AFT:.2f} mm; pod gap: tractor {POD_GAP['tractor']:.1f} mm, pusher {POD_GAP['pusher']:.1f} mm from the hub")

pl = pv.Plotter(shape=(1, 4), window_size=(1600, 420))
for i, c in enumerate(CONFIGS):
    pl.subplot(0, i)
    pl.add_mesh(POD_MESH[c[0]], color="#c9c9c9", smooth_shading=True)
    pl.add_mesh(PROP_MESH[c[1]], color="#3b6ea5", smooth_shading=True)
    pl.add_text(LABEL[c], font_size=10)
    pl.camera_position = [(-0.35, -0.55, 0.30), (0.0, 0.05, 0.0), (0, 1, 0)]
pl.link_views()
aviz.show(pl)

## 2. Aerodynamic efficiency — blade element momentum theory (Boreas)

Propeller efficiency is `η = T V / P`: the thrust power delivered to the aircraft over the shaft power taken from the
motor. At a fixed rpm it is a function of the advance ratio `J = V / (n D)` alone; the left plot is that curve with
the thrust and power coefficients `C_T = T / (ρ n² D⁴)` and `C_P = P / (ρ n³ D⁵)`. A third blade of the same shape adds
a half more solidity: more thrust and power per revolution, peak efficiency at a slightly different `J`, a little
lower peak (more blade area means more profile drag for the same thrust). The right plot is what the aircraft sees:
the efficiency at the thrust its drag polar needs at every speed (level flight), each propeller at its own rpm.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
for B in BLADE_COUNTS:
    rpm = OPS[B]["cruise"].rpm
    n = rpm / 60
    rows = [boreas.solve(PROPS[B], SECTION, rpm, V, RHO) for V in np.linspace(0.5, 1.05 * BLADE["pitch_mm"] / 1000 * n, 40)]
    J = [o.advance_ratio for o in rows]
    col = COLOR[("tractor", B)]                          # one colour per blade count
    ax[0].plot(J, [o.efficiency for o in rows], "-", color=col, lw=2, label=f"η, {B} blades")
    ax[1].plot(J, [o.ct for o in rows], "-", color=col, lw=2, label=f"C_T, {B} blades")
    ax[1].plot(J, [o.cp for o in rows], ":", color=col, lw=2, label=f"C_P, {B} blades")
    op = OPS[B]["cruise"]
    ax[0].plot(op.advance_ratio, op.efficiency, "o", color=col)
    Vs = np.linspace(9, 32, 24)
    eta = [boreas.rpm_for_thrust(PROPS[B], SECTION, required_thrust(V), V, RHO).efficiency for V in Vs]
    ax[2].plot(Vs, eta, "-", color=col, lw=2, label=f"{B} blades")
ax[0].set(xlabel="advance ratio J", ylabel="efficiency η", title="at the cruise rpm (dots: cruise)", ylim=(0, 0.85)); ax[0].legend()
ax[1].set(xlabel="advance ratio J", ylabel="coefficient", title="thrust and power coefficients"); ax[1].legend(fontsize=8)
ax[2].set(xlabel="airspeed [m/s]", ylabel="efficiency η", title="in level flight, at the thrust the aircraft needs"); ax[2].legend()
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for B in BLADE_COUNTS:
    op = OPS[B]["cruise"]
    rR = op.r / PROPS[B].radius
    ls = "-" if B == 2 else "--"
    ax[0].plot(rR, op.dT_dr / B, ls, label=f"{B} blades")
    ax[1].plot(rR, op.alpha_deg, ls, label=f"{B} blades")
    ax[2].plot(rR, op.induced_velocity, ls, label=f"{B} blades")
ax[0].set(xlabel="r / R", ylabel="dT/dr per blade [N/m]", title="thrust along one blade, cruise")
ax[1].set(xlabel="r / R", ylabel="angle of attack [deg]", title="section angle of attack")
ax[2].set(xlabel="r / R", ylabel="induced axial velocity [m/s]", title="what the disc adds to the flow")
for a in ax: a.grid(alpha=0.3); a.legend()
fig.tight_layout()
eff = pd.DataFrame({f"{B} blades": {f"η {k}" if k != "take-off" else "figure of merit, take-off": (op.efficiency if k != "take-off" else op.figure_of_merit)
                                    for k, op in OPS[B].items()} for B in BLADE_COUNTS})
eff.loc["shaft power at cruise [W]"] = [OPS[B]["cruise"].power for B in BLADE_COUNTS]
eff.round(3)

## 3. The air the blades meet

Both layouts are modelled alike (`designs/air_propeller.py: installation_wake`): the **potential flow** of the closed
bodies — the pod with the propeller hub as slender-body line sources, the pylon as a closed source sheet over its chord
and span — plus, behind the bodies only (the pusher), their **viscous** wakes. Upstream of a tractor the blades see the
free stream slowed by about one percent ahead of the pylon's leading edge and more near the hub, ahead of the pod's nose,
wide and smooth. Behind a pusher's pod the blades cut through the pylon's viscous wake: a narrow lane where the air is a
seventh slower (Silverstein's airfoil-wake law at the pylon gap: half its total-head loss), and near the hub through the
pod's own boundary-layer wake and the slowed air behind its closing tail cone. The round, steady parts change the mean inflow (section 6), not
the harmonics. The right plot is what matters for vibration and noise: the Fourier orders of the
wake at 0.7 R — the smooth blockage has almost none above the first few, the narrow wake keeps them all.

In [ ]:
R_MM = BLADE["diameter_mm"] / 2

WAKE_MODEL = {lay: ap.installation_wake(pod_p(lay), R_MM, POINTS["cruise"][0], cd=PYLON_CD, nu=NU) for lay in LAYOUTS}
fig = plt.figure(figsize=(16, 4.5))
axp = fig.add_subplot(1, 3, 1, projection="polar")
for lay in LAYOUTS:
    phi = np.arange(0, 360, 0.5)
    axp.plot(np.radians(phi), 1 - WAKE_MODEL[lay](0.7, phi), label=lay, color="#333333" if lay == "tractor" else "#999999")
axp.set_theta_zero_location("N"); axp.set_ylim(0.6, 1.02); axp.set_title("inflow / flight speed at 0.7 R\n(pylon up, at 0°)"); axp.legend(loc="lower left", fontsize=8)
ax = fig.add_subplot(1, 3, 2)
phi = np.linspace(-30, 30, 400)
for lay in LAYOUTS:
    for rf, ls in ((0.5, ":"), (0.7, "-"), (0.95, "--")):
        ax.plot(phi, WAKE_MODEL[lay](rf, phi % 360), ls, color="#333333" if lay == "tractor" else "#999999", label=f"{lay}, r/R {rf}")
ax.set(xlabel="angle from the pylon [deg]", ylabel="wake fraction w", title="the deficit across the pylon"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
ax = fig.add_subplot(1, 3, 3)
k = np.arange(1, 25)
for i, lay in enumerate(LAYOUTS):
    ax.bar(k + (i - 0.5) * 0.4, WAKE_MODEL[lay].harmonics(0.7, 24), width=0.4, label=lay, color="#333333" if lay == "tractor" else "#999999")
ax.set(xlabel="order (per revolution)", ylabel="amplitude of w", title="wake harmonics at 0.7 R", yscale="log", ylim=(1e-5, 0.1)); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()

## 4. The load on one blade over a revolution

At every angle the blade element solution runs at the local inflow (`boreas.wake.load_harmonics`, quasi-steady); the
Fourier orders of that load are what shakes the blade and what radiates sound. A quasi-steady blade answers a sudden
gust at once; a real one builds its lift over a few chord lengths of travel (Sears), so the higher orders are
attenuated by `1 / sqrt(1 + 2π k)` with the reduced frequency `k = q Ω c / (2 V_rel)` at 0.7 R — applied below
(`sears`), and kept in everything after this section.

In [ ]:
def sears(h, prop, V):
    """A copy of the load harmonics with the blade loads' orders q attenuated by the Sears factor 1/sqrt(1 + 2 pi k_q)."""
    r07 = 0.7 * prop.radius
    c07 = float(np.interp(r07, prop.r, prop.chord))
    omega = h.rpm * 2 * math.pi / 60
    k1 = omega * c07 / (2 * math.hypot(V, omega * r07))
    q = np.arange(len(h.blade_thrust) // 2 + 1)
    f = 1 / np.sqrt(1 + 2 * math.pi * k1 * q)
    T = np.fft.irfft(np.fft.rfft(h.blade_thrust) * f, len(h.blade_thrust))
    Q = np.fft.irfft(np.fft.rfft(h.blade_torque) * f, len(h.blade_torque))
    amps = dict(h.amplitudes, blade_thrust_N=wake._amplitudes(T, np.abs(T).max()), blade_torque_Nm=wake._amplitudes(Q, np.abs(Q).max()))
    return dataclasses.replace(h, blade_thrust=T, blade_torque=Q, amplitudes=amps)

WAKE = dict(WAKE_MODEL)                     # section 5 replaces these with the pod-alone CFD wake in the full preset
def loads(c, point="cruise", wk=None):
    B, op = c[1], OPS[c[1]][point]
    h = wake.load_harmonics(PROPS[B], SECTION, op.rpm, op.airspeed, (wk or WAKE)[c[0]], RHO)
    return sears(h, PROPS[B], op.airspeed)

H = {c: loads(c) for c in CONFIGS}
fig, ax = plt.subplots(1, 2, figsize=(16, 4.2))
for c in CONFIGS:
    h = H[c]
    ax[0].plot(h.theta_deg, h.blade_thrust, color=COLOR[c], label=LABEL[c])
    a = h.amplitudes["blade_thrust_N"]
    ax[1].plot(np.arange(1, 21), a[1:21], "o-", color=COLOR[c], label=LABEL[c])
ax[0].set(xlabel="blade angle from the pylon [deg]", ylabel="thrust on one blade [N]", title="cruise: one blade, one revolution"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="order (x shaft rate)", ylabel="amplitude [N]", yscale="log", title="its harmonics (after Sears)", ylim=(1e-5, 1)); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({LABEL[c]: {"rpm": H[c].rpm, "mean blade thrust [N]": H[c].amplitudes["blade_thrust_N"][0],
                          "peak-to-peak [N]": float(np.ptp(H[c].blade_thrust)),
                          "peak-to-peak / mean": float(np.ptp(H[c].blade_thrust) / H[c].amplitudes["blade_thrust_N"][0]),
                          "1P amplitude [N]": H[c].amplitudes["blade_thrust_N"][1],
                          "shaft thrust at B·P [N]": H[c].amplitudes["shaft_thrust_N"][c[1]],
                          "side force at 1P [N]": float(np.hypot(H[c].amplitudes["side_force_y_N"][1], H[c].amplitudes["side_force_z_N"][1]))}
              for c in CONFIGS}).round(4)

## 5. CFD — isolated and installed propellers (OpenFOAM)

Three kinds of case, each meshed with snappyHexMesh and solved steady (k-ω SST):

- **isolated** (`rotor_mrf`): each propeller alone — its own efficiency, against BEMT.
- **installed** (`rotor_mrf_installed`, new): the propeller in a rotating zone next to the standing pod and pylon,
  forces on each apart. The propeller's efficiency changes because the pod changes its inflow (a pusher swallows the
  pod's slow boundary layer and gains; a tractor's slipstream scrubs the pod and the pylon). The **net efficiency**
  `(T − ΔD) V / P` books the drag the running propeller adds to the pod and pylon against it: `ΔD` is the body's drag
  installed minus its drag without the blades (the case below: same template, mesh, domain and inflow, so the
  subtraction cancels most of the discretisation error; meaningful only in the full preset). Steady MRF freezes the blades at one angle
  to the pylon; the unsteady blade passing needs a sliding mesh, so for the noise the wake comes from:
- **pod alone** (`rotor_mrf_installed` with a **bladeless hub** as the rotor, each layout): the body's drag without the
  propeller (the hub still shields the pod's motor face, as in the installed cases), and the flow on the propeller plane,
  read into a `WakeField` (`boreas.wake.sampled_wake`); in the full preset that replaces the wake model of section 3.

Every propeller case runs at the cruise speed at three rpm — **low, medium, high** (`RPM_POINTS`, × its cruise rpm;
medium is the cruise point): the CFD side of section 6.

In [ ]:
RPM_POINTS = {"low": 0.85, "medium": 1.0, "high": 1.25}      # x each propeller's cruise rpm, at the cruise speed
PRESETS = {
    "smoke": dict(rotor=dict(iterations=60, cells_per_diameter=5.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1,
                             upstream=2.0, downstream=3.0, lateral=2.0),
                  static=dict(static_level=2, static_box_level=1, static_margin=1.0),
                  ),
    "full": dict(rotor=dict(iterations=600, cells_per_diameter=8.0, surface_level=4, near_level=3, rotor_level=3, wake_level=2,
                            upstream=3.0, downstream=6.0, lateral=3.0),
                 static=dict(static_level=3, static_box_level=1, static_margin=1.5),
                 ),
}
PRE = PRESETS[CFD_PRESET]
ENV = aeromant.OpenFOAMEnvironment.detect() if RUN_CFD else aeromant.OpenFOAMEnvironment()

def rotor_params(B, V, rpm=None):
    return dict(rpm=rpm or OPS[B]["cruise"].rpm, airspeed=V, diameter=BLADE["diameter_mm"] / 1000, kinematic_viscosity=NU, density=RHO,
                rotation=1, **PRE["rotor"])

KINDS = ("isolated",) + LAYOUTS
V_CFD = POINTS["cruise"][0]
CASES = {}                                    # key: (kind, blades, rpm factor); factor 1.0 is the cruise point
for kind in KINDS:
    for B in BLADE_COUNTS:
        for f in RPM_POINTS.values():
            rpm = f * OPS[B]["cruise"].rpm
            wd = RUNS / f"cfd_{kind}_{B}_{rpm:.0f}rpm"
            if kind == "isolated":
                CASES[(kind, B, f)] = aeromant.CFDCase("rotor_mrf", PROP_STL[B], rotor_params(B, V_CFD, rpm), wd, geometry_units="mm", environment=ENV)
            else:
                CASES[(kind, B, f)] = aeromant.CFDCase("rotor_mrf_installed", PROP_STL[B], dict(rotor_params(B, V_CFD, rpm), **PRE["static"]), wd,
                                                       geometry_units="mm", environment=ENV, static_geometry=POD_STL[kind])
import cadquery as cq
hub_x = dedalus.Geometry.from_cadquery(cq.Workplane("YZ").circle(HUB["hub_diameter"] / 2).extrude(HUB["hub_height"])
                                       .translate((-HUB["hub_height"] / 2, 0, 0)), name="hub_only_axis_x")
HUB_STL = hub_x.export_stl(RUNS / "cad" / "hub_only_axis_x.stl", tolerance=0.02)
for lay in LAYOUTS:                                  # the bladeless reference: same template, mesh settings, domain (sized on D) and inflow
    CASES[("pod alone", lay)] = aeromant.CFDCase("rotor_mrf_installed", HUB_STL, dict(rotor_params(2, V_CFD), **PRE["static"]),
                                                 RUNS / f"cfd_pod_alone_{lay}", geometry_units="mm", environment=ENV, static_geometry=POD_STL[lay])
for k, case in CASES.items():
    r = case.prepare(overwrite=True)
    assert r.ok, (k, r.messages)
print(f"{len(CASES)} cases prepared ({CFD_PRESET} preset)")
aviz.show(aviz.plot_setup(CASES[("pusher", 3, 1.0)]))

In [ ]:
CFD = {}
if RUN_CFD:
    if CFD_PRESET == "smoke":
        print("SMOKE TEST: coarse meshes and 60 iterations — this proves the cases run; the numbers below are NOT results.\n"
              "Run the notebook with AIR_PROP_CFD=full for the real CFD.")
    for k, case in tqdm(CASES.items(), desc="CFD cases"):
        CFD[k] = case.run(processors=PROCS)
        if not CFD[k].ok:
            print(k, "FAILED:", CFD[k].messages[:3])
    print(f"{sum(r.ok for r in CFD.values())} of {len(CFD)} cases ran; total {sum(r.duration_s for r in CFD.values()) / 60:.1f} min")
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1)")
CFD_OK = {k: r for k, r in CFD.items() if r.ok}

In [ ]:
def bemt_at(B, V, rpm):
    return boreas.solve(PROPS[B], SECTION, rpm, V, RHO)

POD_ALONE_DRAG = {lay: CFD_OK[("pod alone", lay)].metrics["body_drag_N"] for lay in LAYOUTS if ("pod alone", lay) in CFD_OK}

def cfd_eff(kind, B, f):
    """CFD propeller efficiency T V / P and net efficiency (T - dD) V / P of a case (dD = body drag installed - alone), or None."""
    r = CFD_OK.get((kind, B, f))
    if r is None:
        return None
    m = r.metrics
    rpm = m["omega_rad_s"] * 60 / (2 * math.pi)
    eta = m["efficiency"] if m.get("efficiency") is not None else np.nan        # None when the power came out <= 0
    out = {"rpm": rpm, "J": m["advance_ratio"], "thrust": m["thrust_N"], "power": m["power_W"], "eta": eta, "dD": None, "eta_net": np.nan}
    if kind in POD_ALONE_DRAG and m.get("body_drag_N") is not None and m["power_W"] > 0:
        out["dD"] = m["body_drag_N"] - POD_ALONE_DRAG[kind]
        out["eta_net"] = (m["thrust_N"] - out["dD"]) * V_CFD / m["power_W"]
    return out

rows = {}
for k, r in CFD_OK.items():
    if k[0] == "pod alone":
        continue
    kind, B, f = k
    m, e = r.metrics, cfd_eff(*k)
    b = bemt_at(B, V_CFD, e["rpm"])
    point = next(n for n, v in RPM_POINTS.items() if v == f)
    rows[f"{kind}, {B} blades, {point} ({e['rpm']:.0f} rpm)"] = {
        "cells": m.get("mesh_cells"), "converged": m.get("converged"), "J": m["advance_ratio"],
        "thrust BEMT, alone [N]": b.thrust, "thrust CFD [N]": m["thrust_N"], "power BEMT, alone [W]": b.power, "power CFD [W]": m["power_W"],
        "η BEMT, alone": b.efficiency, "η CFD (propeller)": m["efficiency"],
        "body drag installed [N]": m.get("body_drag_N"), "ΔD = installed − alone [N]": e["dD"], "η net (CFD)": e["eta_net"]}
print("pod alone at", V_CFD, "m/s, drag [N]:", {k: round(v, 4) for k, v in POD_ALONE_DRAG.items()})
if POD_ALONE_DRAG:
    print("note: dD = body drag installed - with a bladeless hub: same template and settings; the meshes still differ around the blades\n"
          "(their surface refinement), so read the net efficiency from the full preset, with a mesh study")
CFD_TABLE = pd.DataFrame(rows).T
if CFD_PRESET == "smoke" and len(CFD_TABLE):
    print("SMOKE TEST numbers (coarse, unconverged): they show the pipeline works, not the propellers' performance")
CFD_TABLE.round(3) if len(CFD_TABLE) else "no CFD results"

In [ ]:
if CFD_OK:
    fig, ax = plt.subplots(1, 2, figsize=(16, 4.6))
    for B in BLADE_COUNTS:
        rpm = OPS[B]["cruise"].rpm
        Vs = np.linspace(2, 1.05 * BLADE["pitch_mm"] / 1000 * rpm / 60, 40)
        ls = "-" if B == 2 else "--"
        bem = [bemt_at(B, V, rpm) for V in Vs]
        ax[0].plot([o.advance_ratio for o in bem], [o.efficiency for o in bem], ls, color="#555", label=f"BEMT, {B} blades")
        iso = [cfd_eff("isolated", B, f) for f in RPM_POINTS.values()]
        iso = [e for e in iso if e]
        ax[0].plot([e["J"] for e in iso], [e["eta"] for e in iso], "s" + ls, color="k", mfc="white", label=f"CFD isolated, {B} blades")
        for lay in LAYOUTS:
            es = [e for e in (cfd_eff(lay, B, f) for f in RPM_POINTS.values()) if e]
            mk = "o" if lay == "tractor" else "^"
            ax[0].plot([e["J"] for e in es], [e["eta"] for e in es], mk, color=COLOR[(lay, B)], ms=8, label=f"CFD {lay} {B}: propeller η")
            if any(np.isfinite(e["eta_net"]) for e in es):
                ax[0].plot([e["J"] for e in es], [e["eta_net"] for e in es], mk, color=COLOR[(lay, B)], mfc="none", ms=8, label=f"CFD {lay} {B}: net η")
    ax[0].set(xlabel="advance ratio J", ylabel="efficiency", title=f"efficiency: BEMT and CFD ({CFD_PRESET} preset)", ylim=(0, 1)); ax[0].legend(fontsize=7, ncol=2); ax[0].grid(alpha=0.3)
    have = [c for c in CONFIGS if cfd_eff(c[0], c[1], 1.0)]
    labels = [LABEL[c] for c in have]
    eta_p = [cfd_eff(c[0], c[1], 1.0)["eta"] for c in have]
    eta_n = [cfd_eff(c[0], c[1], 1.0)["eta_net"] or np.nan for c in have]
    x = np.arange(len(labels))
    ax[1].bar(x - 0.2, eta_p, 0.4, label="propeller η = T V / P", color=[COLOR[c] for c in have])
    ax[1].bar(x + 0.2, eta_n, 0.4, label="net η = (T − ΔD) V / P", color="#bbbbbb")
    ax[1].set_xticks(x, labels, rotation=15); ax[1].set(ylabel="efficiency", title="installed at cruise (CFD)" + (" — SMOKE TEST, not results" if CFD_PRESET == "smoke" else ""))
    ax[1].legend(); ax[1].grid(alpha=0.3, axis="y")
    fig.tight_layout()

In [ ]:
# the flow in a section through the axis and the pylon (z = 0): tractor and pusher, three blades
if CFD_OK:
    fig, axs = plt.subplots(1, 2, figsize=(17, 5))
    for ax, lay in zip(axs, LAYOUTS):
        k = (lay, 3, 1.0)
        if k in CFD_OK:
            aviz.plot_section(CASES[k], field="U", normal="z", origin=(0.0, 0.0, 0.0), ax=ax, levels=30)
            for poly in OUTLINE[lay]["side"]:
                ax.fill(poly[:, 0], poly[:, 1], color="#dddddd", ec="#666666", lw=0.8)
            ax.set_xlim(-0.45, 0.6); ax.set_ylim(-0.25, 0.3); ax.set_title(f"{LABEL[(lay, 3)]}: |U| in the plane of the pylon")
    fig.tight_layout()

In [ ]:
# the wake on the propeller plane, read from the pod-alone CFD; in the full preset it replaces the model (section 3)
WAKE_CFD = {}
for lay in LAYOUTS:
    r = CFD_OK.get(("pod alone", lay))
    if r is None:
        continue
    s = movie.openfoam_sampler(CASES[("pod alone", lay)])
    WAKE_CFD[lay] = wake.sampled_wake(s, (0.0, 0.0, 0.0), R_MM / 1000, POINTS["cruise"][0], r_frac=(0.25, 0.4, 0.55, 0.7, 0.85, 0.97), n_phi=180,
                                      source=f"pod-alone CFD ({CFD_PRESET})")
if WAKE_CFD:
    fig = plt.figure(figsize=(12, 4.2))
    for i, lay in enumerate(WAKE_CFD):
        axp = fig.add_subplot(1, 2, i + 1, projection="polar")
        phi = np.arange(0, 360, 1.0)
        axp.plot(np.radians(phi), 1 - WAKE_MODEL[lay](0.7, phi), label="model", color="#555")
        axp.plot(np.radians(phi), 1 - WAKE_CFD[lay](0.7, phi), label=f"CFD ({CFD_PRESET})", color="k")
        axp.set_theta_zero_location("N"); axp.set_ylim(0.5, 1.1); axp.set_title(f"{lay}: inflow / flight speed at 0.7 R"); axp.legend(fontsize=8, loc="lower left")
    fig.tight_layout()
if CFD_PRESET == "full" and len(WAKE_CFD) == len(LAYOUTS):
    WAKE.update(WAKE_CFD)
    H = {c: loads(c) for c in CONFIGS}
    print("the blade loads and everything after this use the CFD wake")
else:
    print("the blade loads and everything after this use the wake model of section 3" + (" (the smoke-test wake is too coarse to use)" if WAKE_CFD else ""))

## 6. Efficiency against rpm — alone, tractor, pusher; two and three blades

Six panels: the rows are the blade count, the columns the installation — the propeller **alone** (the reference),
**tractor**, **pusher**. In each, against rpm:

- **propeller efficiency** `η = T V / P` at five airspeeds (solid lines, light → dark = slow → fast; Boreas BEMT). Installed,
  the propeller works in the installation's mean inflow — the thrust-weighted wake of pod and pylon
  (`boreas.wake.effective_inflow` over `installation_wake`) — and `V` is still the flight speed, so a slower inflow counts
  as a gain;
- **net efficiency** `(T − ΔD) V / P` (dashed, installed panels): `ΔD` is the drag the running propeller adds to the pod
  and the pylon (`installation_drag`: the pressure field of the disc, loaded as the blade-element solution loads it —
  little at the root, where the pod is — on the pod, a tractor's nose pushed back and a pusher's tail cone sucked back,
  and on the pylon's thickness; and the faster air scrubbing pod and pylon); `ΔD / T` is the thrust deduction `t`;
- **where the aircraft flies** (black dots, joined): at each speed the rpm at which the net thrust equals the aircraft's
  drag (section 1's polar);
- **CFD** at the cruise speed over the rpm sweep (squares: propeller η; open squares: net η with `ΔD` from the CFD).

At a fixed speed the efficiency is zero at the rpm where the thrust vanishes — the blades meet the air at their zero-lift
angle, at `J ≈ 0.72`, above the geometric pitch ratio 0.6 because the cambered section still lifts at zero geometric angle
(its zero-lift angle is −3°) — rises to a peak at about 1.3 times that rpm (`J ≈ 0.55`) and falls slowly as the blades are
loaded harder (more induced loss, more profile drag at the higher section speeds).

In [ ]:
SPEEDS = (10.0, 15.0, 20.0, 25.0, 30.0)
SPEED_COLORS = ("#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281")   # one hue, light -> dark: an ordinal ramp (validated)
RPM_MAX = 14500.0
INST_WAKE = {(lay, V): ap.installation_wake(pod_p(lay), R_MM, V, cd=PYLON_CD, nu=NU) for lay in LAYOUTS for V in SPEEDS}

def model_point(kind, B, rpm, V):
    # model operating point of propeller B at rpm and airspeed V, alone or installed (kind = isolated / tractor / pusher)
    if kind == "isolated":
        w, op, dD = 0.0, boreas.solve(PROPS[B], SECTION, rpm, V, RHO), 0.0
    else:
        wk = INST_WAKE.get((kind, V)) or ap.installation_wake(pod_p(kind), R_MM, V, cd=PYLON_CD, nu=NU)
        w, op = wake.effective_inflow(PROPS[B], SECTION, rpm, V, wk, RHO)
        dD = ap.installation_drag(pod_p(kind), R_MM, V, op.thrust, rho=RHO, nu=NU, cd_pylon=PYLON_CD, loading=(op.r, op.dT_dr))["total_N"]
    ok = op.power > 0 and op.thrust > 0
    return {"rpm": rpm, "V": V, "thrust": op.thrust, "net_thrust": op.thrust - dD, "power": op.power, "dD": dD, "w_eff": w,
            "t": dD / op.thrust if op.thrust > 0 else np.nan, "J": V / (rpm / 60 * PROPS[B].diameter),
            "eta": op.thrust * V / op.power if ok else np.nan,
            "eta_net": (op.thrust - dD) * V / op.power if ok and op.thrust > dD else np.nan}

def flying_rpm(kind, B, V, lo=1500.0, hi=20000.0):
    # the rpm at which the net thrust equals the aircraft's drag at V (bisection on the model)
    need = required_thrust(V)
    for _ in range(22):                                                     # ~0.005 rpm
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if model_point(kind, B, mid, V)["net_thrust"] < need else (lo, mid)
    return 0.5 * (lo + hi)

def zero_thrust_rpm(kind, B, V, lo=300.0, hi=20000.0):
    # the rpm at which the propeller (alone or installed) stops making thrust at V: its efficiency is zero there
    for _ in range(22):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if model_point(kind, B, mid, V)["thrust"] <= 0 else (lo, mid)
    return hi

def rpm_grid(kind, B, V, n=44):
    # from the zero-thrust rpm to RPM_MAX, packed near the start where the efficiency climbs steeply
    r0 = zero_thrust_rpm(kind, B, V)
    return r0 + (RPM_MAX - r0) * np.linspace(0.0, 1.0, n) ** 1.8

CURVES = {(kind, B, V): [model_point(kind, B, n, V) for n in rpm_grid(kind, B, V)] for kind in KINDS for B in BLADE_COUNTS for V in SPEEDS}
FLY = {(kind, B, V): model_point(kind, B, flying_rpm(kind, B, V), V) for kind in KINDS for B in BLADE_COUNTS for V in SPEEDS}
print("model curves:", len(CURVES), "of", len(next(iter(CURVES.values()))), "rpm each; operating points:", len(FLY))

In [ ]:
INK, MUTED = "#222222", "#6b6b6b"
DRAWN = {"cfd": False, "cfd_net": False}
TITLE = {"isolated": "alone (no pod)", "tractor": "tractor", "pusher": "pusher"}
fig, axs = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
for i, B in enumerate(BLADE_COUNTS):
    for j, kind in enumerate(KINDS):
        ax = axs[i, j]
        for V, col in zip(SPEEDS, SPEED_COLORS):
            cur = CURVES[(kind, B, V)]
            rpm_ = np.array([c["rpm"] for c in cur])
            eta = np.array([c["eta"] for c in cur])
            ax.plot(rpm_, eta, color=col, lw=2)
            if kind != "isolated":
                ax.plot(rpm_, [c["eta_net"] for c in cur], color=col, lw=1.2, ls="--")
            if np.isfinite(eta).any():                                          # direct label at the peak, in ink
                k = int(np.nanargmax(eta))
                ax.annotate(f"{V:.0f} m/s", (rpm_[k], eta[k]), xytext=(0, 5), textcoords="offset points", ha="center", fontsize=8, color=MUTED)
        fly = [FLY[(kind, B, V)] for V in SPEEDS]
        key = "eta" if kind == "isolated" else "eta_net"
        ax.plot([f["rpm"] for f in fly], [f[key] for f in fly], "-o", color=INK, lw=1, ms=6, mec="white", mew=1.5, zorder=5)
        es = [e for e in (cfd_eff(kind, B, f) for f in RPM_POINTS.values()) if e] if CFD_OK else []
        if es:
            ax.plot([e["rpm"] for e in es], [e["eta"] for e in es], "s", color=INK, ms=8, mec="white", mew=1.5, zorder=6)
            DRAWN["cfd"] = True
            if kind != "isolated" and any(np.isfinite(e["eta_net"]) for e in es):
                ax.plot([e["rpm"] for e in es], [e["eta_net"] for e in es], "s", mfc="white", mec=INK, ms=8, mew=1.3, zorder=6)
                DRAWN["cfd_net"] = True
        ax.set_title(f"{B} blades — {TITLE[kind]}", fontsize=12)
        ax.grid(color="#e8e8e8", lw=0.6); ax.set_axisbelow(True)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
for ax in axs[1]:
    ax.set_xlabel("rpm")
for ax in axs[:, 0]:
    ax.set_ylabel("efficiency")
axs[0, 0].set_ylim(0, 0.85); axs[0, 0].set_xlim(3000, RPM_MAX)
from matplotlib.lines import Line2D
handles = [Line2D([], [], color=c, lw=2, label=f"{V:.0f} m/s") for V, c in zip(SPEEDS, SPEED_COLORS)] + [
    Line2D([], [], color=MUTED, lw=2, label="propeller η = T V / P (model)"),
    Line2D([], [], color=MUTED, lw=1.2, ls="--", label="net η = (T − ΔD) V / P (model, installed)"),
    Line2D([], [], color=INK, lw=1, marker="o", ms=6, mec="white", label="where the aircraft flies (net thrust = drag)")]
if DRAWN["cfd"]:                                                        # only what is drawn; the smoke caveat only when it ran
    handles.append(Line2D([], [], color=INK, ls="", marker="s", ms=8, label=f"CFD propeller η at {V_CFD:.0f} m/s" + (" — smoke test, not results" if CFD_PRESET == "smoke" else "")))
if DRAWN["cfd_net"]:
    handles.append(Line2D([], [], color=INK, ls="", marker="s", mfc="white", ms=8, label="CFD net η (installed)"))
fig.suptitle("Efficiency against rpm — 10 × 6 inch propeller, alone / tractor / pusher, two and three blades", y=0.995, fontsize=14)
fig.legend(handles=handles, loc="upper center", ncol=5, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 0.97))
fig.tight_layout(rect=(0, 0, 1, 0.925))
fig.savefig(OUT / "efficiency_vs_rpm.png", dpi=130, bbox_inches="tight")

In [ ]:
# the numbers behind the panels, at the cruise speed: the peak, where the aircraft flies, the installation's share
V = POINTS["cruise"][0]
rows_ = {}
for kind in KINDS:
    for B in BLADE_COUNTS:
        cur = CURVES[(kind, B, V)]
        eta = np.array([c["eta"] for c in cur])
        k = int(np.nanargmax(eta))
        rpm_peak = cur[k]["rpm"]
        fly = FLY[(kind, B, V)]
        e = cfd_eff(kind, B, 1.0) if CFD_OK else None
        rows_[(kind, f"{B} blades")] = {"peak η (model)": eta[k], "at rpm": rpm_peak, "rpm where it flies": fly["rpm"],
                                        "η where it flies": fly["eta"], "η net where it flies": fly["eta_net"] if kind != "isolated" else fly["eta"],
                                        "effective wake w": fly["w_eff"], "thrust deduction t": fly["t"] if kind != "isolated" else 0.0,
                                        "shaft power where it flies [W]": fly["power"],
                                        f"CFD η, cruise rpm ({CFD_PRESET})": e["eta"] if e else np.nan,
                                        f"CFD net η, cruise rpm ({CFD_PRESET})": (e["eta_net"] if e and e["eta_net"] is not None else np.nan)}
EFF_TABLE = pd.DataFrame(rows_).T
display(EFF_TABLE.round(4))
alone = {B: EFF_TABLE.loc[("isolated", f"{B} blades"), "η net where it flies"] for B in BLADE_COUNTS}
for B in BLADE_COUNTS:
    d_tp = {Vv: FLY[("pusher", B, Vv)]["eta_net"] - FLY[("tractor", B, Vv)]["eta_net"] for Vv in SPEEDS}
    print(f"{B} blades, pusher minus tractor (net η where it flies, points): " + ", ".join(f"{Vv:.0f} m/s {100 * d:+.2f}" for Vv, d in d_tp.items()))
    print(f"{B} blades at {V:.0f} m/s, net efficiency where it flies: alone {alone[B]:.3f}, "
          + ", ".join(f"{lay} {EFF_TABLE.loc[(lay, f'{B} blades'), 'η net where it flies']:.3f} "
                      f"({100 * (EFF_TABLE.loc[(lay, f'{B} blades'), 'η net where it flies'] - alone[B]):+.1f} points)" for lay in LAYOUTS))

# where the installation's share comes from: the mean wake along the blade and the thrust that weights it
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
rf = np.linspace(0.06, 1.0, 120)
for lay, col in (("tractor", "#333333"), ("pusher", "#999999")):
    ax[0].plot(rf, [WAKE_MODEL[lay].mean(x) for x in rf], color=col, lw=2, label=f"{lay}: pod + pylon, circumferential mean")
ax[0].set(xlabel="r / R", ylabel="mean wake fraction w̄", title=f"the mean inflow deficit along the blade at {V:.0f} m/s", yscale="log", ylim=(1e-4, 1)); ax[0].legend(fontsize=8)
for B, ls in zip(BLADE_COUNTS, ("-", "--")):
    op = OPS[B]["cruise"]
    ax[1].plot(op.r / PROPS[B].radius, op.dT_dr / np.max(op.dT_dr), ls, color=INK, label=f"{B} blades: dT/dr (normalised), the weights")
ax[1].set(xlabel="r / R", ylabel="relative thrust", title="where the blade makes its thrust (cruise)"); ax[1].legend(fontsize=8)
for a in ax:
    a.grid(color="#e8e8e8", lw=0.6)
fig.tight_layout()

## 7. The flow in 3D, with particles

Streamlines and a cloud of tracer balls (each with a short trail), coloured by speed, through the propeller and past
the pod — the CFD field of the installed pusher and tractor when the CFD has run, otherwise the slipstream model of the
blade-element solution (no pod). Rotate and zoom in a live kernel.

In [ ]:
def field_grid(sampler, D, n=(64, 30, 30), x_range=(-1.8, 2.6), lateral=0.9):
    lo = np.array([x_range[0] * D, -lateral * D, -lateral * D])
    sp = np.array([(x_range[1] - x_range[0]) * D / (n[0] - 1), 2 * lateral * D / (n[1] - 1), 2 * lateral * D / (n[2] - 1)])
    grid = pv.ImageData(dimensions=n, spacing=tuple(sp), origin=tuple(lo))
    U, ok = sampler(np.asarray(grid.points))
    grid.point_data["U"] = np.where(ok[:, None], U, 0.0)
    grid.point_data["|U|"] = np.linalg.norm(grid.point_data["U"], axis=1)
    return grid

def particle_scene(sampler, rotor, title, layout=None, n=180, steps=80, degrees_per_frame=8):
    D = rotor.diameter
    view = dict(feed_upstream=1.6, lateral=1.0) if layout == "pusher" else {}
    tr = movie.Tracer(sampler, rotor, n=n, seed=3, trail=14, **view)
    dt = movie.time_step(rotor, degrees_per_frame)
    for _ in range(steps):
        tr.step(dt)
    grid = field_grid(sampler, D)
    seeds = pv.Disc(center=(-1.5 * D, 0, 0), inner=0.08 * D, outer=0.6 * D, normal=(1, 0, 0), r_res=4, c_res=18)
    stream = grid.streamlines_from_source(seeds, vectors="U", max_steps=4000)
    vmax = float(np.percentile(grid.point_data["|U|"][grid.point_data["|U|"] > 0], 99))
    pl = pv.Plotter(window_size=(960, 560))
    if stream.n_points:
        pl.add_mesh(stream.tube(radius=0.003 * D), scalars="|U|", cmap="turbo", clim=(0, vmax), opacity=0.35, show_scalar_bar=False)
    L = tr.trails.shape[0]
    pts_ = tr.trails.transpose(1, 0, 2).reshape(-1, 3)
    lines = np.hstack([[L] + list(range(j * L, (j + 1) * L)) for j in range(tr.n)])
    trails = pv.PolyData(pts_, lines=lines); trails.point_data["|U|"] = np.repeat(tr.speed, L)
    pl.add_mesh(trails, scalars="|U|", cmap="turbo", clim=(0, vmax), line_width=2, show_scalar_bar=False)
    cloud = pv.PolyData(tr.pos); cloud.point_data["|U|"] = tr.speed
    pl.add_mesh(cloud, scalars="|U|", cmap="turbo", clim=(0, vmax), render_points_as_spheres=True, point_size=11, scalar_bar_args={"title": "|U| [m/s]"})
    pl.add_mesh(PROP_MESH[rotor.blades], color="#3b6ea5", smooth_shading=True)
    if layout:
        pl.add_mesh(POD_MESH[layout], color="#c9c9c9", smooth_shading=True, opacity=0.9)
    pl.add_text(title, font_size=10); pl.add_axes()
    pl.camera_position = [(-0.35, 0.45, 0.85), (0.05, 0.03, 0.0), (0, 1, 0)]
    return pl

def model_field(B, point="cruise"):
    op = OPS[B][point]
    rot = movie.RotorView(center=(0.0, 0.0, 0.0), diameter=PROPS[B].diameter, rpm=op.rpm, rotation=1, blades=B, inflow=op.airspeed)
    return wake.slipstream_sampler(PROPS[B], op), rot

SAMPLERS = {}
for c in CONFIGS:
    k = (c[0], c[1], 1.0)
    if k in CFD_OK:
        SAMPLERS[c] = (movie.openfoam_sampler(CASES[k]), movie.RotorView.from_case(CASES[k], c[1]), f"OpenFOAM ({CFD_PRESET})")
    else:
        s, r = model_field(c[1])
        SAMPLERS[c] = (s, r, "slipstream model")
for c in (("tractor", 3), ("pusher", 3)):
    s, r, src = SAMPLERS[c]
    aviz.show(particle_scene(s, r, f"{LABEL[c]} at cruise, {r.rpm:.0f} rpm — {src}", layout=c[0] if src.startswith("OpenFOAM") else None))

## 8. Movies: tracer balls through the four configurations

A few dozen tracer balls carried by the flow, a side view (with the pod and the pylon) and a view along the axis
(the blades turning, the swirl), in consistent slow motion: every frame the propeller turns 10° and the air moves
on by the same time (`vegeta.aeromant.movie`). One clip per configuration — tractor with two blades, tractor with three,
pusher with two, pusher with three — joined into one movie. From the CFD when it ran (smoke preset: a coarse field,
the pictures are right in kind, not in detail), otherwise from the slipstream model.

In [ ]:
clips = []
for c in CONFIGS:
    s, r, src = SAMPLERS[c]
    on_cfd = src.startswith("OpenFOAM")
    view = dict(feed_upstream=1.7, lateral=1.0, downstream=2.6) if c[0] == "pusher" else dict(feed_upstream=1.2, lateral=1.0, downstream=3.0)
    clips.append(movie.make_movie(None, OUT / f"flow_{c[0]}_{c[1]}.mp4", rotor=r, sampler=s, n=60, seconds=5, fps=24, degrees_per_frame=10,
                                  outline=OUTLINE[c[0]] if on_cfd else None, view=view,
                                  title=f"{LABEL[c]}: {r.inflow:.0f} m/s, {r.rpm:.0f} rpm ({src})"))
FLOW_MOVIE = movie.concat_videos(clips, OUT / "flow_tractor_then_pusher.mp4", captions=[LABEL[c] for c in CONFIGS])
display(Video(str(FLOW_MOVIE), embed=False, width=960))

## 9. The blade in FEA (Talos): spinning and loaded

One blade with its hub, held at the shaft bore. Two unit load cases — the **spin** (`talos.Centrifugal`: the
centrifugal body force ρ ω² r at the take-off rpm) and the **air load** (the blade's share of the take-off thrust
and torque as tractions over the blade) — and their superposition at any operating point (linear: the spin stress
scales with rpm², the air load with the thrust). The two- and three-blade blades are the same blade; the three-blade
propeller turns slower and loads each blade less for the same thrust. Then the modes (Talos `*FREQUENCY`), with the
flap mode stiffened by the spin (Southwell: `f² = f₀² + K n²`, K = 1.2 assumed), against the engine orders.

In [ ]:
BLADE_MAT = talos.Material(MATERIAL["name"], youngs_modulus=MATERIAL["E"], poissons_ratio=MATERIAL["nu"], density=MATERIAL["rho"],
                           yield_strength=MATERIAL["yield_"], source="assumed moulded glass-filled nylon")
blade_cad = PropellerCAD().generate(**cad_kw(1))
blade_step = blade_cad.export(RUNS / "cad_blade", formats=("step",)).artifacts["step"]
rb, hh, Rm = HUB["bore"] / 2 + 0.5, HUB["hub_height"], BLADE["diameter_mm"] / 2
REGIONS = [talos.SurfacesInBox("bore", (-rb, -rb, -hh / 2 - 0.5, rb, rb, hh / 2 + 0.5)),
           talos.SurfacesInBox("blade", (HUB["hub_diameter"] / 2 - 2.0, -Rm, -Rm, Rm + 1.0, Rm, Rm))]
MESH = talos.MeshSettings(element_size=1.6)
RPM_REF = max(OPS[B]["take-off"].rpm for B in BLADE_COUNTS)                                # the spin unit case: the highest take-off rpm
T_REF = 1.0                                                                                # the air unit case: 1 N of thrust per blade ...
Q_PER_T = OPS[2]["take-off"].torque / OPS[2]["take-off"].thrust                            # ... with the torque that goes with it
F_TAN_REF = T_REF * Q_PER_T / (0.7 * Rm / 1000)                                            # tangential force at 0.7 R [N]

def fea(name, loads):
    m = talos.StructuralModel(blade_step, "mm-N-MPa", BLADE_MAT, REGIONS, [talos.FixedSupport("bore")], loads, MESH, name=name)
    m.mesh(RUNS / f"fea_{name}")
    return m, m.solve(RUNS / f"fea_{name}")

m_spin, r_spin = fea("spin", [talos.Centrifugal(RPM_REF, point=(0, 0, 0), axis=(0, 0, 1))])
m_air, r_air = fea("air", [talos.Force("blade", fz=T_REF, fy=-F_TAN_REF)])
shutil.copytree(RUNS / "fea_air", RUNS / "fea_modes", dirs_exist_ok=True)
modes = m_air.solve_modes(RUNS / "fea_modes", n_modes=4)
print(r_spin); print(r_air); print(modes)
FR_SPIN, FR_AIR = talos.read_frd(r_spin.artifacts["frd"]), talos.read_frd(r_air.artifacts["frd"])
assert np.array_equal(FR_SPIN.node_ids, FR_AIR.node_ids)
blade_mass_g = blade_cad.volume * MATERIAL["rho"] * 1e6
print(f"one blade with the hub: {blade_mass_g:.1f} g")

In [ ]:
def stress_at(B, point):
    """Stress tensor field (N, 6) of a B-blade propeller's blade at an operating point: spin x (rpm/RPM_REF)^2 + air x thrust per blade."""
    op = OPS[B][point]
    return FR_SPIN.stress * (op.rpm / RPM_REF) ** 2 + FR_AIR.stress * (op.thrust / B / T_REF), op

def weak_points(vm, n=4, spacing=8.0):
    order = np.argsort(-vm)
    picked = []
    for i in order:
        if all(np.linalg.norm(FR_AIR.coords[i] - FR_AIR.coords[j]) > spacing for j in picked):
            picked.append(i)
        if len(picked) == n:
            break
    return picked

FEA_ROWS, WEAK = {}, {}
for B in BLADE_COUNTS:
    for point in ("cruise", "climb", "take-off"):
        S, op = stress_at(B, point)
        vm = talos.frd.von_mises(S)
        i = int(np.argmax(vm))
        r = math.hypot(*FR_AIR.coords[i][:2])
        FEA_ROWS[(f"{B} blades", point)] = {"rpm": op.rpm, "thrust per blade [N]": op.thrust / B,
                                         "max von Mises [MPa]": vm[i], "of which spin alone [MPa]": float(talos.frd.von_mises(FR_SPIN.stress * (op.rpm / RPM_REF) ** 2)[i]),
                                         "at r / R": r / Rm, "SF yield": MATERIAL["yield_"] / vm[i]}
    S, _ = stress_at(B, "take-off")
    WEAK[B] = weak_points(talos.frd.von_mises(S))
display(pd.DataFrame(FEA_ROWS).T.round(3))
weak_tab = pd.DataFrame([{"blades": B, "point": f"W{j + 1}", "r/R": math.hypot(*FR_AIR.coords[i][:2]) / Rm,
                          "z [mm] (+ = suction face)": FR_AIR.coords[i][2], "von Mises at take-off [MPa]": float(talos.frd.von_mises(stress_at(B, "take-off")[0])[i])}
                         for B in BLADE_COUNTS for j, i in enumerate(WEAK[B])]).set_index(["blades", "point"])
weak_tab.round(2)

In [ ]:
# the combined take-off stress of the 2-blade blade, its weak points, and the stress movie (the load ramps up while it turns)
def combined(B, point):
    S, op = stress_at(B, point)
    disp = FR_SPIN.displacement * (op.rpm / RPM_REF) ** 2 + FR_AIR.displacement * (op.thrust / B / T_REF)
    return talos.FieldResults(FR_AIR.node_ids, FR_AIR.coords, {"DISP": disp, "STRESS": S}), op

B = 2
FR_TO, op = combined(B, "take-off")
pl = tviz.plot_results(FR_TO, field="von_mises", mesh=r_air.artifacts["mesh"])
hot = FR_AIR.coords[WEAK[B]]
pl.add_mesh(pv.PolyData(hot), color="#d50000", render_points_as_spheres=True, point_size=16)
pl.add_point_labels(hot + np.array([0, 0, 6.0]), [f"W{j + 1}" for j in range(len(hot))], font_size=14, point_size=1, shape_opacity=0.6)
aviz.show(pl)
tviz.show(tviz.plot_mode(modes, mode=1))
STRESS_MOVIE = tviz.animate(FR_TO, OUT / "blade_stress_take_off.mp4", mesh=r_air.artifacts["mesh"], rpm=op.rpm, axis="z", seconds=5, fps=24)
display(Video(str(STRESS_MOVIE), embed=False, width=720))

In [ ]:
f0 = np.array(modes.metrics["frequencies_hz"])
K_SOUTHWELL = np.array([1.2, 0.0, 0.0, 0.0])[: len(f0)]          # spin stiffening: flap mode only (assumed); others conservatively none
rpm_axis = np.linspace(0, 1.1 * RPM_REF, 200)
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for i, f in enumerate(f0):
    ax[0].plot(rpm_axis, np.sqrt(f**2 + K_SOUTHWELL[i] * (rpm_axis / 60) ** 2), lw=2, label=f"mode {i + 1} ({f:.0f} Hz at rest)")
for q in range(1, 9):
    ax[0].plot(rpm_axis, q * rpm_axis / 60, color="#999", lw=0.8, ls=":" if q not in BLADE_COUNTS else "-")
    ax[0].text(rpm_axis[-1], q * rpm_axis[-1] / 60, f"{q}P", fontsize=8, va="center")
for B, ls in zip(BLADE_COUNTS, ("-", "--")):
    lo, hi = min(op.rpm for op in OPS[B].values()), max(op.rpm for op in OPS[B].values())
    ax[0].axvspan(lo, hi, color="#2ca02c" if B == 2 else "#9467bd", alpha=0.12, label=f"{B} blades: cruise .. take-off rpm")
ax[0].set(xlabel="rpm", ylabel="frequency [Hz]", title="Campbell diagram of the blade", ylim=(0, 1.1 * max(f0.max(), 8 * RPM_REF / 60))); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
margin = {}
for c in CONFIGS:
    op = OPS[c[1]]["cruise"]
    fm = np.sqrt(f0**2 + K_SOUTHWELL * (op.rpm / 60) ** 2)
    a = H[c].amplitudes["blade_thrust_N"]
    sig = [q for q in range(1, 13) if a[q] > 0.01 * a[0]]
    margin[LABEL[c]] = min((abs(fm[i] - q * op.rpm / 60) / fm[i] for i in range(len(fm)) for q in sig), default=np.nan)
ax[1].bar(range(len(margin)), list(margin.values()), color=[COLOR[c] for c in CONFIGS])
ax[1].axhline(0.15, color="#c62828", ls="--", label="15 % margin")
ax[1].set_xticks(range(len(margin)), list(margin), rotation=15); ax[1].set(ylabel="nearest mode to a significant order", title="frequency margin at cruise"); ax[1].legend(); ax[1].grid(alpha=0.3, axis="y")
fig.tight_layout()

## 10. Vibration and fatigue: the once-per-revolution pulse

Every revolution the pusher's blade crosses the pylon wake: its thrust drops and recovers, an alternating stress on
top of the steady (spin + mean air load) stress — about 10⁷ cycles in 5 hours of flight. Goodman's line with the
material's fatigue strength gives the safety factor at each weak point; the amplification of the nearest mode is
included (damping ratio `MATERIAL["damping"]`, plus 0.02 aerodynamic).

In [ ]:
ZETA = MATERIAL["damping"] + 0.02
def goodman(c, point="cruise"):
    B = c[1]
    h = loads(c, point)
    S_mean, op = stress_at(B, point)
    fm = np.sqrt(f0**2 + K_SOUTHWELL * (op.rpm / 60) ** 2)
    a = h.amplitudes["blade_thrust_N"]
    amp = 0.0                                   # alternating thrust, every order amplified by the nearest mode (summed: an upper bound)
    for q in range(1, 25):
        r = q * op.rpm / 60 / fm
        daf = np.max(1 / np.sqrt((1 - r**2) ** 2 + (2 * ZETA * r) ** 2))
        amp += a[q] * daf
    out = []
    for i in WEAK[B]:
        sm = float(talos.frd.von_mises(S_mean[i:i + 1])[0])
        sa = float(talos.frd.von_mises(FR_AIR.stress[i:i + 1])[0]) * amp / T_REF
        sf = 1 / (sa / MATERIAL["fatigue_1e7"] + sm / MATERIAL["ultimate"])
        out.append((sm, sa, sf))
    k = int(np.argmin([o[2] for o in out]))
    return {"alternating blade thrust [N]": amp, "mean stress [MPa]": out[k][0], "alternating stress [MPa]": out[k][1], "Goodman SF": out[k][2]}

FATIGUE = pd.DataFrame({LABEL[c]: {**{f"cruise: {k}": v for k, v in goodman(c, "cruise").items()},
                                   **{f"climb: {k}": v for k, v in goodman(c, "climb").items()}} for c in CONFIGS})
FATIGUE.round(3)

## 11. How loud

Two parts, at 10 m from the propeller:

- **tones of the blade loading** at multiples of the blade-passing frequency (BPF = B × rpm / 60), from the blade's
  load around the revolution as rotating dipoles (`boreas.wake.rotating_tones`, Lowson): the steady load gives Gutin's
  tones, the wake's load pulses add to them — for the pusher strongly, and mostly **towards the axis** (ahead of and
  behind the aircraft), where the steady tones are weak;
- **broadband vortex noise** (trailing edges, turbulence), Schlegel–King–Mull (`boreas.vortex_noise`), peaking at a few kHz;
- the levels in **dB** and **dB(A)** (A-weighting, `boreas.a_weighting`: what a sound-level meter and an ear make of it).

The directivity matters here more than anywhere: in the disc plane (90°) the steady tones dominate and tractor and
pusher sound alike; towards the axis the tractor goes quiet and the pusher does not. So the table gives the level in the
disc plane, at 45° from the axis (an aircraft approaching or leaving), and in the **loudest direction** — the peak a
listener hears as the aircraft flies over. Thickness noise is not modelled (at tip Mach numbers below 0.45 it is
smaller than the loading tones, and the same for both layouts); the observer is below the aircraft (azimuth 180° from the pylon).
At take-off (no flight speed) the wake model has nothing to carry the pylon's wake into the disc — the real pusher still
draws its own inflow past the pylon — so there the two layouts come out alike; read the take-off point as the steady tones only.

In [ ]:
DIST = 10.0
NOISE_AZ = 180.0                                # observer below: opposite the pylon
LISTEN_ANGLE = 45.0                             # the spectra and the sounds: 45 deg from the axis
ANGLES = np.arange(10, 171, 5)

def noise(c, point="cruise", angle=LISTEN_ANGLE, distance=DIST, harmonics=16):
    B, op = c[1], OPS[c[1]][point]
    h = loads(c, point)
    tones = wake.rotating_tones(h, PROPS[B], distance, AIR, angle, harmonics=harmonics, azimuth_deg=NOISE_AZ)
    steady = wake.rotating_tones(loads(("tractor", B), point, {"tractor": wake.uniform_wake(0.0)}), PROPS[B], distance, AIR, angle,
                                 harmonics=harmonics, azimuth_deg=NOISE_AZ)
    vn = boreas.vortex_noise(PROPS[B], op.rpm, distance, op.airspeed, thickness_ratio=BLADE["thickness"])
    band = (vn["peak_hz"] / 2, min(vn["peak_hz"] * 2, 20000.0))
    bb_a = vn["spl_db"] + 10 * math.log10(np.mean(10 ** (boreas.a_weighting(np.geomspace(*band, 200)) / 10)))
    f = np.array([t["frequency_hz"] for t in tones]); L = np.array([t["spl_db"] for t in tones])
    tonal, tonal_a = boreas.add_levels(L), boreas.add_levels(L + boreas.a_weighting(f))
    return dict(tones=tones, steady=steady, broadband=vn["spl_db"], band=band, tonal=tonal, tonal_A=tonal_a,
                total=boreas.add_levels([tonal, vn["spl_db"]]), total_A=boreas.add_levels([tonal_a, bb_a]),
                bpf=B * op.rpm / 60, loudest=float(L.max()))

NOISE = {c: noise(c) for c in CONFIGS}
DIRECTIVITY = {c: np.array([noise(c, angle=a)["total_A"] for a in ANGLES]) for c in CONFIGS}
fig, axs = plt.subplots(1, 4, figsize=(19, 4.3), sharey=True)
for ax, c in zip(axs, CONFIGS):
    n_ = NOISE[c]
    f = [t["frequency_hz"] for t in n_["tones"]]
    ax.stem(f, [t["spl_db"] for t in n_["tones"]], linefmt=COLOR[c], markerfmt="o", basefmt=" ", label="steady + wake")
    ax.plot([t["frequency_hz"] for t in n_["steady"]], [t["spl_db"] for t in n_["steady"]], "x", color="k", label="steady only (Gutin)")
    ax.hlines(n_["broadband"] - 10 * math.log10(3 * math.log2(n_["band"][1] / n_["band"][0])), *n_["band"], color="#888", ls="--",
              label="vortex noise (per third octave)")
    ax.set(xscale="log", xlabel="frequency [Hz]", title=f"{LABEL[c]}, {LISTEN_ANGLE:.0f}° from the axis\n{n_['total']:.1f} dB, {n_['total_A']:.1f} dB(A) at {DIST:.0f} m", ylim=(0, 70))
    ax.grid(alpha=0.3, which="both")
axs[0].set_ylabel(f"dB re 20 µPa at {DIST:.0f} m"); axs[0].legend(fontsize=8)
fig.tight_layout()
LOUD = pd.DataFrame({LABEL[c]: {"BPF [Hz]": NOISE[c]["bpf"],
                                "in the disc plane (90°) [dB(A)]": noise(c, angle=90.0)["total_A"],
                                f"at {LISTEN_ANGLE:.0f}° from the axis [dB(A)]": NOISE[c]["total_A"],
                                "loudest direction [dB(A)]": float(DIRECTIVITY[c].max()),
                                "... at [deg from the axis]": float(ANGLES[np.argmax(DIRECTIVITY[c])]),
                                "tones at 45° [dB]": NOISE[c]["tonal"], "vortex noise [dB]": NOISE[c]["broadband"],
                                "loudest direction at 100 m [dB(A)]": float(DIRECTIVITY[c].max()) - 20} for c in CONFIGS})
LOUD.round(1)

In [ ]:
fig = plt.figure(figsize=(16, 4.6))
axp = fig.add_subplot(1, 2, 1, projection="polar")
for c in CONFIGS:
    th = np.radians(ANGLES)
    axp.plot(th, DIRECTIVITY[c], "-" if c[1] == 2 else "--", color=COLOR[c], label=LABEL[c])
axp.set_thetamin(0); axp.set_thetamax(180); axp.set_theta_zero_location("N"); axp.set_theta_direction(-1)
axp.set_title(f"directivity at cruise, dB(A) at {DIST:.0f} m\n(0° = ahead along the axis, 180° = behind)"); axp.legend(fontsize=8, loc="lower left")
ax = fig.add_subplot(1, 2, 2)
for c in CONFIGS:
    ls = "-" if c[1] == 2 else "--"
    ax.plot(range(3), [max(noise(c, p, angle=a)["total_A"] for a in (20, 45, 90, 135, 160)) for p in ("cruise", "climb", "take-off")],
            "o" + ls, color=COLOR[c], label=LABEL[c])
ax.set_xticks(range(3), ["cruise", "climb", "take-off"]); ax.set(ylabel=f"loudest direction, dB(A) at {DIST:.0f} m", title="by flight point"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()

### Listen

The cruise sound of each configuration at 10 m and 45° from the axis (tones and broadband synthesised from the levels above), all four
files on **one** scale: the louder propeller is louder on playback. Then the four one after the other.

In [ ]:
RATE = 44100
SIG = {c: ap.synthesize([(t["frequency_hz"], t["spl_db"]) for t in NOISE[c]["tones"]], NOISE[c]["broadband"], seconds=3.0, rate=RATE,
                        band=NOISE[c]["band"], seed=i) for i, c in enumerate(CONFIGS)}
FULL = 1.2 * max(np.abs(s).max() for s in SIG.values())
WAV = {c: ap.write_wav(OUT / f"sound_{c[0]}_{c[1]}.wav", SIG[c], FULL, RATE) for c in CONFIGS}
gap = np.zeros(int(0.6 * RATE))
ALL_WAV = ap.write_wav(OUT / "sound_tractor_then_pusher.wav", np.concatenate(sum(([SIG[c], gap] for c in CONFIGS), [])), FULL, RATE)
for c in CONFIGS:
    print(f"{LABEL[c]}: {NOISE[c]['total_A']:.1f} dB(A) at {LISTEN_ANGLE:.0f}° — {WAV[c]}")
    display(Audio(url=str(WAV[c])))                 # linked, not embedded (like the movies)
print("all four, tractor first (embedded):")
display(Audio(str(ALL_WAV)))

## 12. Side by side, and export

In [ ]:
def cfd_metric(c, key):
    e = cfd_eff(c[0], c[1], 1.0)
    return e[key] if e is not None and e[key] is not None else np.nan

SUMMARY = pd.DataFrame({LABEL[c]: {
    "cruise rpm (installed, where it flies)": EFF_TABLE.loc[(c[0], f"{c[1]} blades"), "rpm where it flies"],
    "η cruise, BEMT": OPS[c[1]]["cruise"].efficiency,
    "η net at the cruise operating point (model)": EFF_TABLE.loc[(c[0], f"{c[1]} blades"), "η net where it flies"],
    f"η propeller, CFD ({CFD_PRESET})": cfd_metric(c, "eta"),
    f"η net, CFD ({CFD_PRESET})": cfd_metric(c, "eta_net"),
    "figure of merit, take-off": OPS[c[1]]["take-off"].figure_of_merit,
    "blade thrust pulse (p-p / mean)": float(np.ptp(H[c].blade_thrust) / H[c].amplitudes["blade_thrust_N"][0]),
    "SF yield, take-off": FEA_ROWS[(f"{c[1]} blades", "take-off")]["SF yield"],
    "Goodman SF, climb": FATIGUE.loc["climb: Goodman SF", LABEL[c]],
    "dB(A) at 10 m, disc plane": LOUD.loc["in the disc plane (90°) [dB(A)]", LABEL[c]],
    "dB(A) at 10 m, loudest direction": LOUD.loc["loudest direction [dB(A)]", LABEL[c]]} for c in CONFIGS})
display(SUMMARY.round(3))
fig, axs = plt.subplots(1, 4, figsize=(19, 3.8))
for ax, (key, better) in zip(axs, (("η net at the cruise operating point (model)", "higher"), ("Goodman SF, climb", "higher"), ("dB(A) at 10 m, disc plane", "lower"), ("dB(A) at 10 m, loudest direction", "lower"))):
    ax.bar(range(len(CONFIGS)), SUMMARY.loc[key].astype(float), color=[COLOR[c] for c in CONFIGS])
    ax.set_xticks(range(len(CONFIGS)), [LABEL[c].replace(", ", "\n") for c in CONFIGS], fontsize=8)
    ax.set_title(f"{key} ({better} is better)", fontsize=10); ax.grid(alpha=0.3, axis="y")
    lo, hi = SUMMARY.loc[key].astype(float).min(), SUMMARY.loc[key].astype(float).max()
    ax.set_ylim(lo - 0.6 * (hi - lo + 1e-9) if key.startswith("η") or "dB" in key else 0, hi + 0.15 * (hi - lo + 1e-9))
fig.tight_layout()
peak = {c: float(DIRECTIVITY[c].max()) for c in CONFIGS}
quiet, loud = min(CONFIGS, key=peak.get), max(CONFIGS, key=peak.get)
print(f"loudest direction at {DIST:.0f} m: quietest {LABEL[quiet]} ({peak[quiet]:.1f} dB(A)), loudest {LABEL[loud]} ({peak[loud]:.1f} dB(A)), "
      f"{peak[loud] - peak[quiet]:.1f} dB(A) apart")
for B in BLADE_COUNTS:
    d = peak[("pusher", B)] - peak[("tractor", B)]
    d90 = LOUD.loc["in the disc plane (90°) [dB(A)]", LABEL[("pusher", B)]] - LOUD.loc["in the disc plane (90°) [dB(A)]", LABEL[("tractor", B)]]
    print(f"{B} blades: the pusher is {d:+.1f} dB(A) against the tractor in the loudest direction, {d90:+.1f} dB(A) in the disc plane")

In [ ]:
doc = {"blade": BLADE, "hub": HUB, "section": dataclasses.asdict(SECTION), "pod": POD, "pylon_cd": PYLON_CD, "aircraft": AIRCRAFT, "material": MATERIAL,
       "operating_points": {f"{B} blades": {k: op.to_dict() for k, op in OPS[B].items()} for B in BLADE_COUNTS},
       "wake": {lay: {"source": WAKE[lay].source, "harmonics_0.7R": WAKE[lay].harmonics(0.7, 16).tolist()} for lay in LAYOUTS},
       "cfd": {"preset": CFD_PRESET, "smoke_test_numbers_are_not_results": CFD_PRESET == "smoke",
               "cases": {" / ".join(map(str, k)): {kk: vv for kk, vv in r.metrics.items() if not isinstance(vv, (list, dict))} for k, r in CFD_OK.items()}},
       "fea": {"spin_rpm_ref": RPM_REF, "modes_hz_at_rest": f0.tolist(), "southwell_K": K_SOUTHWELL.tolist(),
               "by_point": {f"{a} / {b}": v for (a, b), v in FEA_ROWS.items()}},
       "fatigue": FATIGUE.to_dict(), "noise_10m": LOUD.to_dict(), "summary": SUMMARY.to_dict(),
       "files": {"flow_movie": str(FLOW_MOVIE), "sounds": {LABEL[c]: str(WAV[c]) for c in CONFIGS}, "sound_all": str(ALL_WAV)}}
(RUNS / "air_propeller.json").write_text(json.dumps(doc, indent=2, default=float))
print("written", RUNS / "air_propeller.json", "| movies and sounds in", OUT)

**Reading the result.** Efficiency first (section 2): at the same thrust the three-blade propeller turns slower and
its tip speed is lower; its cruise efficiency is about the same as the two-blade one (a little lower: more blade area
for the same thrust), its static figure of merit higher — it is the better climber, the two-blade the better cruiser.
Against rpm (section 6) every propeller has the same shape: no efficiency at the rpm where the thrust vanishes, a peak
a little above it, a slow decline as the blades are loaded harder — and the aircraft flies to the right of the peak at
low speed, near it at high speed. At this size (a 46 mm pod in a 254 mm disc) the installation moves the net efficiency
where the aircraft flies by only a few tenths of a point: the tractor's blades work in air slowed by the pod's nose and
the hub, and push extra drag onto the pod and the pylon; the pusher's work in the pod's and pylon's slower wake — a real
gain, the recovered wake — and suck on the tail cone and the pylon. In the model the pusher comes out a few tenths of a
point ahead of the tractor (section 6 prints the numbers) — small enough that only the full CFD (sections 5 and 6) can
confirm it; read the CFD numbers only from the **full** preset, the smoke test only shows that the cases run.

Loads, strength and sound (sections 4, 9–11): the tractor's blades meet clean air, their load is nearly steady,
their sound is the Gutin tones at the blade-passing frequency, loud in the disc plane and weak towards the axis. The
pusher's blades cross the pylon's wake once a revolution: a sharp thrust pulse on every blade, a vibration at every
engine order (check the Campbell diagram for a mode on one of them), an alternating stress that lowers the fatigue
safety factor, and a whole comb of blade-passing harmonics radiated mostly along the axis — in the disc plane the two
layouts sound alike, ahead of and behind the aircraft the pusher is much louder (the directivity plot and the
loudest-direction column). A third blade lowers the rpm and the steady tones (the tractor gets quieter); behind a
pylon it hardly helps, because the wake tones do not care much about the blade count.

Vibration (section 9): the blade's first flap mode, stiffened by the spin (Southwell, K = 1.2 assumed), runs close to
the 1P line over the whole rpm range — normal for a light, flexible blade, harmless while the blade sees a nearly steady
load (tractor), and the reason the pusher's margin bar is small: its wake drives 1P strongly. Check it with a measured
spin-up (a strobe or an accelerometer on the motor mount) before trusting the assumed K. The highest static stress is
at the blade–hub junction, a sharp corner in the CAD: a root fillet is the first change it calls for. To quieten a pusher: more gap between pylon and disc (the wake fills in), a thinner
or more distant pylon, a blade count that puts its tones where the wake harmonics are weak, a swept (skewed) blade.

## 13. Export for the comparison notebooks

The ducted fan (`25b_ducted_fan`) and the six- and twelve-blade propellers (`25c_exotic_propellers`) have their own
notebooks; they compare themselves with the two- and three-blade propellers of this one by **reading this export** —
`designs/data/air_propeller_results.json` — instead of computing them again (without it they show their own results only).
It holds the design inputs (blade, section, aircraft, flight points, material, noise setup), and per propeller the
operating points, the efficiency and power in level flight, the full-throttle thrust against airspeed, the efficiency
against rpm at the five airspeeds, and per configuration the installation, noise, blade strength and fatigue numbers and
the files (sounds, flow movies). What did not run here (no OpenFOAM, no CalculiX) is left out.

In [ ]:
RESULTS_JSON = Path("designs/data/air_propeller_results.json")

def opt(f):
    """The value of f(), or None when the part of the notebook it needs did not run."""
    try:
        v = f()
        return None if v is None or (isinstance(v, float) and not np.isfinite(v)) else v
    except Exception:
        return None

VS_EXPORT = np.linspace(9.0, 32.0, 24)
VA_EXPORT = np.linspace(0.0, 40.0, 81)
props_out = {}
for B in BLADE_COUNTS:
    lvl = [boreas.rpm_for_thrust(PROPS[B], SECTION, required_thrust(V), V, RHO) for V in VS_EXPORT]
    to_rpm = OPS[B]["take-off"].rpm
    full = [boreas.solve(PROPS[B], SECTION, to_rpm, V, RHO) for V in VA_EXPORT]
    props_out[str(B)] = {
        "label": f"10 x 6, {B} blades", "operating_points": {k: op.to_dict() for k, op in OPS[B].items()},
        "level": {"V": VS_EXPORT.tolist(), "eta": [o.efficiency for o in lvl], "power_W": [o.power for o in lvl], "rpm": [o.rpm for o in lvl]},
        "full_throttle": {"rpm": to_rpm, "V": VA_EXPORT.tolist(), "thrust_N": [o.thrust for o in full], "power_W": [o.power for o in full]},
        "eta_vs_rpm_alone": {f"{V:g}": [[c["rpm"], c["eta"]] for c in CURVES[("isolated", B, V)]] for V in SPEEDS},
        "rotor_mass_kg": {2: 0.045, 3: 0.050}.get(B),
        "blade_sf_yield_take_off": opt(lambda: FEA_ROWS[(f"{B} blades", "take-off")]["SF yield"]),
    }
P_REF_SHAFT, TIP_MACH_LIMIT = 1900.0 * 0.85, 0.75        # the comparison notebooks' full power: 1900 W electrical, tip Mach 0.75
V_FULL = (0.5, 10.0, 20.0, 30.0, 40.0, 50.0)

def full_power_installed(lay, B, V):
    """The installed propeller at airspeed V at P_REF_SHAFT (or its tip-Mach rpm, whichever comes first)."""
    tip = TIP_MACH_LIMIT * 340.0 / (math.pi * PROPS[B].diameter) * 60
    hi = model_point(lay, B, tip, V)
    if hi["power"] <= P_REF_SHAFT:
        return hi
    lo_, hi_ = 500.0, tip
    for _ in range(18):
        mid = 0.5 * (lo_ + hi_)
        lo_, hi_ = (mid, hi_) if model_point(lay, B, mid, V)["power"] <= P_REF_SHAFT else (lo_, mid)
    return model_point(lay, B, lo_, V)

KEEP = ("rpm", "V", "thrust", "net_thrust", "power", "w_eff", "t", "eta", "eta_net")
configs_out = {}
for c in CONFIGS:
    lay, B = c
    configs_out[f"{lay} {B}"] = {
        "label": LABEL[c], "layout": lay, "blades": B,
        "eta_net_where_it_flies": opt(lambda: EFF_TABLE.loc[(lay, f"{B} blades"), "η net where it flies"]),
        "effective_wake": opt(lambda: EFF_TABLE.loc[(lay, f"{B} blades"), "effective wake w"]),
        "thrust_deduction": opt(lambda: EFF_TABLE.loc[(lay, f"{B} blades"), "thrust deduction t"]),
        "bpf_hz": opt(lambda: NOISE[c]["bpf"]), "dBA_listen": opt(lambda: NOISE[c]["total_A"]),
        "dBA_disc_plane": opt(lambda: LOUD.loc["in the disc plane (90°) [dB(A)]", LABEL[c]]),
        "directivity_dBA": opt(lambda: DIRECTIVITY[c].tolist()),
        "goodman_sf_climb": opt(lambda: FATIGUE.loc["climb: Goodman SF", LABEL[c]]),
        "wav": opt(lambda: str(WAV[c])), "flow_movie": str(OUT / f"flow_{lay}_{B}.mp4") if (OUT / f"flow_{lay}_{B}.mp4").exists() else None,
        "level_flight": [{k: FLY[(lay, B, V)][k] for k in KEEP} for V in SPEEDS],          # where the aircraft flies, installed
        "full_power": [{k: p_[k] for k in KEEP} for p_ in (full_power_installed(lay, B, V) for V in V_FULL)],
    }
EXPORT = {"source": "25_air_propeller", "blade": BLADE, "hub": HUB, "section": dataclasses.asdict(SECTION), "aircraft": AIRCRAFT,
          "pod": POD, "pod_gap_mm": POD_GAP, "pylon_cd": PYLON_CD, "full_power_reference": {"shaft_W": P_REF_SHAFT, "tip_mach": TIP_MACH_LIMIT},
          "points": {k: list(v) for k, v in POINTS.items()}, "material": MATERIAL, "speeds": list(SPEEDS), "rho": RHO, "nu": NU,
          "noise_setup": {"distance_m": opt(lambda: DIST), "listen_angle_deg": opt(lambda: LISTEN_ANGLE), "angles_deg": opt(lambda: ANGLES.tolist()),
                          "rate": opt(lambda: RATE), "full_scale_pa": opt(lambda: float(FULL))},
          "propellers": props_out, "configs": configs_out, "cfd_preset": CFD_PRESET}
RESULTS_JSON.parent.mkdir(parents=True, exist_ok=True)
RESULTS_JSON.write_text(json.dumps(EXPORT, indent=1, default=float))
print("written", RESULTS_JSON, f"({RESULTS_JSON.stat().st_size / 1e3:.0f} kB):", ", ".join(LABEL[c] for c in CONFIGS))

## 14. The propeller maps for the flight models

MERLIN's mission and race (notebooks 26b and 27) do not solve propellers: they read maps. Here every two- and three-blade
propeller of a design space — 9–12 inch, pitch 0.6–1.2 diameters, this notebook's blade and section — is solved over
airspeed × rpm in the free stream, with its installation on MERLIN as a tractor and as a pusher (the effective wake and the
thrust deduction from `installation_wake` and `installation_drag`, MERLIN's fuselage as the pod, its wing or tail as the
pylon), and written to `designs/data/propeller_maps.json` (`designs/propulsor_maps.py`; about a minute on four cores). The
committed file is loaded unless `AIR_PROP_MAPS=build`.

In [ ]:
import propulsor_maps as pmaps
if os.environ.get("AIR_PROP_MAPS") == "build" or not pmaps.LIBRARIES["propellers"].exists():
    print("written", pmaps.save(pmaps.build_library("propellers", progress=True)))
PMAPS = pmaps.load(pmaps.LIBRARIES["propellers"])
rows = {}
for e in PMAPS["entries"]:
    if "error" in e:
        continue
    for lay in ("tractor", "pusher"):
        u = pmaps.unit(e, 1900.0, layout=lay)
        rows[(e["label"], lay)] = {"static thrust at 1900 W [N]": u.full(0.0)[0], "at 30 m/s [N]": u.full(30.0)[0], "at 60 m/s [N]": u.full(60.0)[0],
                                   "w": e["installation"][lay]["w"], "t": e["installation"][lay]["t"]}
MAP_TAB = pd.DataFrame(rows).T
print(f"{len(PMAPS['entries'])} propeller maps in {pmaps.LIBRARIES['propellers']}")
MAP_TAB.sort_values("at 60 m/s [N]", ascending=False).head(12).round(3)